# 04_Model_Evaluation

## Purpose

This notebook evaluates the trained Hybrid ViT–Swin classifier on the test set. It performs inference, computes evaluation metrics, generates the confusion matrix, and plots ROC curves with Macro and Weighted ROC-AUC scores.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_curve,
    roc_auc_score,
    balanced_accuracy_score,
    cohen_kappa_score
)
from sklearn.preprocessing import label_binarize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_NAMES = [
    "Adbhuta","Bhayanaka","Bibhatsya",
    "Hasya","Karuna","Raudra",
    "Shanta","Shringara","Veera"
]
NUM_CLASSES = len(CLASS_NAMES)


In [ ]:
BASE_DIR = "/content/drive/MyDrive/NAVARASA_PROJECT_DRIVE1"
FEATURE_DIR = os.path.join(BASE_DIR,"extracted_features")
MODEL_DIR = os.path.join(BASE_DIR,"models","hybrid")


In [ ]:
test_data = torch.load(os.path.join(FEATURE_DIR,"test.pt"))
X_swin_te = test_data["X_swin"]
X_vit_te = test_data["X_vit"]
y_te = test_data["y"]

from torch.utils.data import TensorDataset, DataLoader
test_dl = DataLoader(TensorDataset(X_swin_te,X_vit_te,y_te),batch_size=64)


In [ ]:
class HybridClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(1024+768,512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512,NUM_CLASSES)
        )
    def forward(self,s,v):
        return self.fc(torch.cat([s,v],dim=1))

model = HybridClassifier().to(device)
model.load_state_dict(torch.load(os.path.join(MODEL_DIR,"best_hybrid.pt"), map_location=device))
model.eval()


In [ ]:
y_true, y_pred, y_prob = [], [], []

with torch.no_grad():
    for s,v,y in test_dl:
        prob = F.softmax(model(s.to(device),v.to(device)), dim=1)
        y_true.extend(y.numpy())
        y_pred.extend(prob.argmax(1).cpu().numpy())
        y_prob.extend(prob.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

print("Accuracy:", accuracy_score(y_true,y_pred))


In [ ]:
prec, rec, f1, sup = precision_recall_fscore_support(
    y_true, y_pred, zero_division=0
)

df = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Precision": prec,
    "Recall": rec,
    "F1-score": f1,
    "Support": sup
})

display(df.round(4))

print("Balanced Accuracy:", round(balanced_accuracy_score(y_true,y_pred),4))
print("Cohen's Kappa   :", round(cohen_kappa_score(y_true,y_pred),4))


In [ ]:
cm = confusion_matrix(y_true,y_pred)

plt.figure(figsize=(9,7))
sns.heatmap(cm,annot=True,fmt="d",
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES,
            cmap="Purples")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix – Hybrid Model")
plt.show()


In [ ]:
plt.figure(figsize=(8,6))

for i, cls in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_true==i, y_prob[:,i])
    auc_score = roc_auc_score(y_true==i, y_prob[:,i])
    plt.plot(fpr,tpr,label=f"{cls} (AUC={auc_score:.2f})")

plt.plot([0,1],[0,1],"k--")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curves – Hybrid Model")
plt.legend(fontsize=8)
plt.show()

y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))

macro_auc = roc_auc_score(
    y_true_bin,
    y_prob,
    average="macro",
    multi_class="ovr"
)

weighted_auc = roc_auc_score(
    y_true_bin,
    y_prob,
    average="weighted",
    multi_class="ovr"
)

print(f"Macro AUC    : {macro_auc:.4f}")
print(f"Weighted AUC : {weighted_auc:.4f}")
